In [ ]:
include("CustomAwid/src/CustomAwid.jl")
using .CustomAwid

using MLDatasets
import Flux
train_data = MLDatasets.FashionMNIST(split=:train)
test_data  = MLDatasets.FashionMNIST(split=:test)

function loader(data; batchsize::Int=1)
    x4dim = reshape(data.features, 28, 28, 1, :) # 3 wymiar to kanał: 1 (czarno-biały)
    yhot  = Flux.onehotbatch(data.targets, 0:9)  # 10×60000 OneHotMatrix
    Flux.DataLoader((x4dim, yhot); batchsize, shuffle=true)
end

In [ ]:
function evaluate_accuracy(model_graph, data_loader, input_node, target_node, output_node)
    correct = 0
    total = 0
    
    # We use Flux's onecold to easily grab the index of the highest probability
    for (x, y) in data_loader
        # Skip incomplete batches at the tail end of the dataset
        if size(x, 4) != size(input_node.data, 4)
            continue
        end
        
        # Forward pass without backward or optimize
        forward!(model_graph, input_node => x, target_node => y)
        
        # onecold converts the one-hot encoded matrix back to class indices (0-9)
        pred = Flux.onecold(output_node.data, 0:9)
        truth = Flux.onecold(y, 0:9)
        
        correct += sum(pred .== truth)
        total += length(truth)
    end
    
    return correct / total
end

In [ ]:
epochs = 3
η = 0.01
batch_size = 10

println("\n[x] Ładowanie danych...")
train_loader = loader(train_data; batchsize=batch_size)
test_loader  = loader(test_data; batchsize=batch_size)

println("[x] Budowanie architektury sieci neuronowej...")
net = chain((
  conv((3, 3), 1 => 6, pad=1, bias=false),
  maxpool((2, 2)),
  conv((3, 3), 6 => 16, pad=1, bias=false),
  maxpool((2, 2)),
  flatten(),
  dense(784 => 84, relu),
  dropout(0.4),
  dense(84 => 10)
))

input  = tensor(28, 28, 1, batch_size) 
target = tensor(10, batch_size)        
output = net(input) # Zwraca przedostatni GraphNode (przed lossem)
loss   = logitcrossentropy(output, target) # zwraca ostatni GraphNode (loss)
model  = graph(loss) # zwraca wektor GraphNode'ów w odpowiedniej kolejności

In [ ]:
println("\n[x] Rozpoczynanie uczenia...")
accuracy = zeros(epochs, 2) # dla 3 epok: 3x2

for epoch in 1:epochs
    println("--- Epoka $epoch ---")
    
    @time begin
        for (x, y) in train_loader
            # Pomijanie niepełnych batchy (gdyby liczba danych nie była podzielna przez liczbę danych)
            if size(x, 4) != batch_size
                continue 
            end
            
            zerograd!(model) # zerowanie gradientów
            forward!(model, input => x, target => y) # przejście w przód
            backward!(model) # przejście w tył
            optimize!(model, η) # poprawienie wag (spadek gradientu)
        end
    end
    
    train_acc = evaluate_accuracy(model, train_loader, input, target, output)
    test_acc  = evaluate_accuracy(model, test_loader, input, target, output)
    
    train_acc_pct = round(train_acc * 100, digits=2)
    test_acc_pct  = round(test_acc * 100, digits=2)
    
    println("Dokładność na zbiorze uczącym: $train_acc_pct%")
    println("Dokładność na zbiorze testowym:  $test_acc_pct%\n")
    
    accuracy[epoch, 1] = train_acc_pct
    accuracy[epoch, 2] = test_acc_pct
end

In [ ]:
println("[x] Generowanie wykresów...")
using CairoMakie

fig = Figure()
gca = Axis(fig[1,1], xlabel="epoka (-)", ylabel="dokładność (%)")
hlines!(85, linestyle=:dash, color=:black, label="85% Target")

lines!(gca, 1:epochs, accuracy[:,1], linewidth=2, label="Uczący")
lines!(gca, 1:epochs, accuracy[:,2], linewidth=2, label="Testowy")
ylims!(gca, (0, 100))
axislegend(gca, position = :rb)

display(fig)